# Controlled RoCoLe leakage injection
Injected scores are biased by design. Supply the original split and local images before running.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from uuid import uuid4
import hashlib
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image, ImageEnhance
from sklearn.metrics import f1_score

DATA_ROOT = Path('data')
SPLIT_CSV = DATA_ROOT / 'combined_group_split.csv'
split = pd.read_csv(SPLIT_CSV)
train, val, test = [split.loc[split.split.eq(s)].reset_index(drop=True)
                    for s in ('train', 'val', 'test')]
classes = ['healthy', 'rust']
assert len(split) == 1389 and split.image_path.is_unique
for frame, counts in zip((train, val, test), ((550, 420), (127, 81), (112, 99))):
    assert tuple(frame.binary_class.value_counts().reindex(classes)) == counts
for path in split.image_path:
    assert not Path(path).is_absolute() and '..' not in Path(path).parts
    assert (DATA_ROOT / path).is_file(), path

def decode(path, label):
    image = tf.image.decode_jpeg(tf.io.read_file(path), channels=3)
    image = tf.image.resize(image, (224, 224))
    return tf.keras.applications.mobilenet_v2.preprocess_input(image), tf.cast(label, tf.float32)

def dataset(frame, seed=None):
    paths = [str(DATA_ROOT / p) for p in frame.image_path]
    labels = frame.binary_class.eq('rust').astype(int).tolist()
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if seed is not None:
        ds = ds.shuffle(len(frame), seed=seed, reshuffle_each_iteration=True)
    return ds.map(decode, num_parallel_calls=tf.data.AUTOTUNE).batch(32).prefetch(tf.data.AUTOTUNE)

def build_model(seed):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    base = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
    base.trainable = False
    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = tf.keras.layers.GlobalAveragePooling2D()(base(inputs, training=False))
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    model = tf.keras.Model(inputs, tf.keras.layers.Dense(1, activation='sigmoid')(x))
    model.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss='binary_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])
    return model

def run_seed(seed, out):
    rng = np.random.default_rng(seed)
    orders = {c: rng.permutation(train.index[train.binary_class.eq(c)]) for c in classes}
    donors = {c: rng.permutation(test.index[test.binary_class.eq(c)]) for c in classes}
    truth = test.binary_class.eq('rust').to_numpy(dtype=int)
    val_ds, test_ds = dataset(val), dataset(test)
    rows, digests = [], set()
    with TemporaryDirectory() as tmp:
        variants = []
        for c in classes:
            for rank, pos in enumerate(orders[c][:len(orders[c]) * 40 // 100]):
                source = test.loc[donors[c][rank % len(donors[c])], 'image_path']
                angle = float(rng.choice([-1, 1]) * rng.uniform(1, 3))
                brightness = float(rng.uniform(0.95, 1.05))
                target = Path(tmp) / f'{c}_{rank:04}.jpg'
                with Image.open(DATA_ROOT / source) as original:
                    image = original.convert('RGB').rotate(angle, resample=Image.Resampling.BILINEAR)
                    w, h = image.size
                    dx, dy = round(w * 0.04), round(h * 0.04)
                    image = image.crop((dx, dy, w - dx, h - dy))
                    ImageEnhance.Brightness(image).enhance(brightness).save(target, quality=92)
                variants.append((c, rank, pos, source, target, angle, brightness))
        for dose in (0, 10, 20, 40):
            folder = out / f'seed_{seed}' / f'dose_{dose:02}'
            folder.mkdir(parents=True, exist_ok=False)
            frame, manifest = train.copy(), []
            for c, rank, pos, source, target, angle, brightness in variants:
                if rank < len(orders[c]) * dose // 100:
                    frame.at[pos, 'image_path'] = str(target.resolve())
                    manifest.append((pos, train.at[pos, 'image_path'], source, angle, brightness))
            pd.DataFrame(manifest, columns=['slot','replaced_image','test_donor','rotation','brightness']).to_csv(folder / 'replacements.csv', index=False)
            model = build_model(seed)
            digests.add(hashlib.sha256(b''.join(w.tobytes() for w in model.get_weights())).hexdigest())
            assert len(digests) == 1
            checkpoint = folder / 'best.keras'
            history = model.fit(dataset(frame, seed), validation_data=val_ds, epochs=10, verbose=2,
                callbacks=[tf.keras.callbacks.ModelCheckpoint(str(checkpoint), monitor='val_loss', mode='min', save_best_only=True),
                           tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)])
            pd.DataFrame(history.history).to_csv(folder / 'history.csv', index=False)
            prob = tf.keras.models.load_model(checkpoint).predict(test_ds, verbose=0).ravel()
            pred = (prob >= 0.5).astype(int)
            conf, correct = np.maximum(prob, 1 - prob), pred == truth
            ece = 0.0
            edges = np.linspace(0, 1, 11)
            for lo, hi in zip(edges[:-1], edges[1:]):
                mask = (conf > lo) & (conf <= hi)
                if mask.any():
                    ece += mask.mean() * abs(correct[mask].mean() - conf[mask].mean())
            sources = {record[2] for record in manifest}
            test.assign(y_true=truth, y_pred=pred, rust_probability=prob,
                        direct_donor=test.image_path.isin(sources)).to_csv(folder / 'predictions.csv', index=False)
            rows.append(dict(seed=seed, dose=dose, accuracy=correct.mean(),
                             macro_f1=f1_score(truth, pred, average='macro'), ece=ece))
            pd.DataFrame(rows).to_csv(folder.parent / 'metrics.csv', index=False)
    return rows


In [ ]:
out = Path('runs') / uuid4().hex
rows = []
for seed in (42, 137, 911):
    rows.extend(run_seed(seed, out))

scores = pd.DataFrame(rows)
keys = ['accuracy', 'macro_f1', 'ece']
baseline = scores.loc[scores.dose.eq(0)].set_index('seed')[keys]
paired = scores.loc[scores.dose.ne(0)].join(baseline, on='seed', rsuffix='_control')
for key in keys:
    paired[key + '_pp'] = 100 * (paired[key] - paired[key + '_control'])
paired.to_csv(out / 'paired_by_seed.csv', index=False)
paired.groupby('dose')[[key + '_pp' for key in keys]].agg(['mean','std','count']).to_csv(out / 'paired_summary.csv')
groups = []
for seed in (42, 137, 911):
    path = out / f'seed_{seed}'
    control = pd.read_csv(path / 'dose_00/predictions.csv').set_index('image_path', verify_integrity=True)
    for dose in (10, 20):
        exposed = pd.read_csv(path / f'dose_{dose:02}/predictions.csv').set_index('image_path', verify_integrity=True)
        comparison = exposed.join(control[['y_true','y_pred']], rsuffix='_control', validate='one_to_one')
        assert len(comparison) == len(control) and comparison.y_true.eq(comparison.y_true_control).all()
        for donor, subset in comparison.groupby('direct_donor'):
            delta = subset.y_pred.eq(subset.y_true).mean() - subset.y_pred_control.eq(subset.y_true).mean()
            groups.append(dict(seed=seed, dose=dose, donor=donor, photos=len(subset), accuracy_delta_pp=100 * delta))
donors = pd.DataFrame(groups)
donors.to_csv(out / 'donor_by_seed.csv', index=False)
donors.groupby(['dose','donor']).accuracy_delta_pp.agg(['mean','std','count']).to_csv(out / 'donor_summary.csv')
print(out)
